In [ ]:
ecommerce/
├── order_service/
│   ├── manage.py
│   ├── orders/
│   │   ├── models.py
│   │   ├── views.py
│   │   ├── services.py
│   │   ├── kafka_producer.py
│   │   ├── kafka_consumer.py
│   │   └── urls.py
│
├── inventory_service/
│   ├── manage.py
│   ├── inventory/
│   │   ├── models.py
│   │   ├── services.py
│   │   ├── kafka_consumer.py
│   │   ├── kafka_producer.py
│
├── payment_service/
│   ├── manage.py
│   ├── payments/
│   │   ├── services.py
│   │   ├── kafka_consumer.py
│   │   ├── kafka_producer.py
│
└── kafka/
    └── topics:
        ├── order-events
        ├── inventory-events
        └── payment-events


In [ ]:
# kafka_config.py
KAFKA_CONF = {
    "bootstrap.servers": "localhost:9092"
}

In [ ]:
# orders/models.py
import uuid
from django.db import models

class Order(models.Model):
    STATUS = [
        ("CREATED", "CREATED"),
        ("PAID", "PAID"),
        ("CANCELLED", "CANCELLED"),
    ]

    id = models.UUIDField(primary_key=True, default=uuid.uuid4)
    user_id = models.UUIDField()
    amount = models.DecimalField(max_digits=10, decimal_places=2)
    status = models.CharField(max_length=20, choices=STATUS)




In [ ]:
# orders/kafka_producer.py
import json, uuid
from confluent_kafka import Producer
from kafka_config import KAFKA_CONF

producer = Producer(KAFKA_CONF)

def publish_order_created(order):
    event = {
        "event_id": str(uuid.uuid4()),
        "event_type": "ORDER_CREATED",
        "payload": {
            "order_id": str(order.id),
            "user_id": str(order.user_id),
            "amount": float(order.amount)
        }
    }

    producer.produce(
        topic="order-events",
        key=str(order.id),
        value=json.dumps(event)
    )
    producer.flush()


In [ ]:
# orders/services.py
from .models import Order
from .kafka_producer import publish_order_created

def create_order(user_id, amount):
    order = Order.objects.create(
        user_id=user_id,
        amount=amount,
        status="CREATED"
    )
    publish_order_created(order)
    return order


In [ ]:
# orders/views.py
from rest_framework.views import APIView
from rest_framework.response import Response
from .services import create_order

class OrderCreateView(APIView):
    def post(self, request):
        order = create_order(
            request.data["user_id"],
            request.data["amount"]
        )
        return Response({"order_id": order.id, "status": order.status})


In [ ]:
# orders/kafka_consumer.py
import json, django
django.setup()

from confluent_kafka import Consumer
from orders.models import Order

consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "order-group",
    "auto.offset.reset": "earliest"
})

consumer.subscribe(["payment-events", "inventory-events"])

while True:
    msg = consumer.poll(1.0)
    if not msg or msg.error():
        continue

    event = json.loads(msg.value())
    order_id = event["payload"]["order_id"]

    if event["event_type"] == "PAYMENT_FAILED":
        Order.objects.filter(id=order_id).update(status="CANCELLED")

    if event["event_type"] == "PAYMENT_SUCCESS":
        Order.objects.filter(id=order_id).update(status="PAID")


In [ ]:
# inventory/models.py
class Inventory(models.Model):
    product_id = models.UUIDField()
    quantity = models.IntegerField()


In [ ]:
# inventory/kafka_consumer.py
import json
from confluent_kafka import Consumer
from .kafka_producer import publish_stock_event

consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "inventory-group",
    "auto.offset.reset": "earliest"
})

consumer.subscribe(["order-events"])

while True:
    msg = consumer.poll(1.0)
    if not msg or msg.error():
        continue

    event = json.loads(msg.value())
    publish_stock_event(event["payload"]["order_id"], success=True)


In [ ]:
# inventory/kafka_producer.py
import json
from confluent_kafka import Producer
from kafka_config import KAFKA_CONF

producer = Producer(KAFKA_CONF)

def publish_stock_event(order_id, success):
    producer.produce(
        topic="inventory-events",
        value=json.dumps({
            "event_type": "STOCK_RESERVED" if success else "STOCK_FAILED",
            "payload": {"order_id": order_id}
        })
    )
    producer.flush()


In [ ]:
# payments/kafka_consumer.py
import json
from confluent_kafka import Consumer
from .kafka_producer import publish_payment_event

consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "payment-group",
    "auto.offset.reset": "earliest"
})

consumer.subscribe(["order-events"])

while True:
    msg = consumer.poll(1.0)
    if not msg or msg.error():
        continue

    event = json.loads(msg.value())
    publish_payment_event(event["payload"]["order_id"], success=True)


In [ ]:
# payments/kafka_producer.py
import json
from confluent_kafka import Producer
from kafka_config import KAFKA_CONF

producer = Producer(KAFKA_CONF)

def publish_payment_event(order_id, success):
    producer.produce(
        topic="payment-events",
        value=json.dumps({
            "event_type": "PAYMENT_SUCCESS" if success else "PAYMENT_FAILED",
            "payload": {"order_id": order_id}
        })
    )
    producer.flush()


In [ ]:
Order Created
→ ORDER_CREATED (Kafka)
→ Inventory + Payment consume
→ STOCK / PAYMENT events
→ Order updates status
